In [1]:
!pip install pillow langchain-openai dotenv pandas datasets


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from os import getenv
import pandas as pd
from pandas import DataFrame
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
import asyncio

In [3]:
from huggingface_hub import login
login()

In [4]:
# Load environment variables and dataset
load_dotenv()
ds: DatasetDict = load_dataset("cais/hle")
ds = ds["test"]

In [5]:
ds = ds.filter(lambda row: row["image_preview"]==None and row["rationale_image"]==None)

Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [6]:
df: DataFrame = ds.to_pandas()
df_noimg = df.drop(["image", "image_preview", "rationale_image"], axis=1)

In [7]:
min_count = df_noimg["category"].value_counts().min()

In [8]:
df_noimg_balanced = df.groupby("category", group_keys=False).apply(lambda x: x.sample(min_count, random_state=42))

C:\Users\devse\AppData\Local\Temp\ipykernel_14608\968111084.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_noimg_balanced = df.groupby("category", group_keys=False).apply(lambda x: x.sample(min_count, random_state=42))


In [9]:
df_noimg_balanced["category"].value_counts()

category
Biology/Medicine             57
Chemistry                    57
Computer Science/AI          57
Engineering                  57
Humanities/Social Science    57
Math                         57
Other                        57
Physics                      57
Name: count, dtype: int64

In [11]:
llm=ChatOpenAI(base_url=getenv("OPENROUTER_BASE_URL") or "https://openrouter.ai/api/v1", api_key=getenv("OPENROUTER_API_KEY"), model="openai/gpt-4.1")

In [12]:
template = """Create a description of this question that is free of details 
that reveal the exact question content, such a terms, numbers, and people.
Make it very general. Respond with just the description.
 
{question}
"""
questions = [template.format(question=q) for q in df_noimg_balanced["question"]]
async def get_descriptions(questions):
    responses = await llm.abatch(questions, config={"max_concurrency": 40})
    print(responses[0])
    return [res.content for res in responses]
df_noimg_balanced["description"] = await get_descriptions(questions)
# def generate_description(row):
#     global askllm
#     if askllm == True:
#         print(llm.invoke("""Create a description of this question that is free of details 
#                that reveal the exact question content, such a terms, numbers, and people.
#                Make it very general. Respond with just the description.
               
#                {question}
#                """.format(question=row["question"])).content)
#         askllm = False
#     return row["question"]

content='This question asks about research findings on the connections between certain biological markers and specific measurement tools used to assess a medical condition in newborns. It involves identifying the type and direction of relationships observed between these markers and assessment outcomes.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 144, 'total_tokens': 189, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'openai/gpt-4.1', 'system_fingerprint': None, 'id': 'gen-1760460322-P8P5SRu5aW1X1dNb23eH', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--ccac776f-111e-4379-81be-55d4bf7cdd29-0' usage_metadata={'input_tokens': 144, 'output_tokens': 45, 'total_tokens': 189, 'input_token_details': {}, 'output_token_details': {}}


In [18]:
Dataset.from_pandas(df_noimg_balanced).save_to_disk("dataset")

Saving the dataset (0/1 shards):   0%|          | 0/456 [00:00<?, ? examples/s]

In [28]:
ds: DatasetDict = load_from_disk("dataset")
ds

Dataset({
    features: ['id', 'question', 'image', 'image_preview', 'answer', 'answer_type', 'author_name', 'rationale', 'rationale_image', 'raw_subject', 'category', 'canary', 'description', '__index_level_0__'],
    num_rows: 456
})

In [29]:
ds.select_columns(['question', 'answer', 'description'])

Dataset({
    features: ['question', 'answer', 'description'],
    num_rows: 456
})